# Delphi: End‑to‑End Training & Evaluation (Colab)

This notebook trains and evaluates Delphi on the provided synthetic dataset.
It includes a full data pipeline, training loop, evaluation metrics (loss + token accuracy),
plots, and inference examples.

**Note:** The demo dataset is synthetic and meant for testing only.


## 1) Setup
Run this cell to clone the repo (if needed) and install dependencies.


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/gerstung-lab/Delphi.git'
if not Path('delphi_torch').exists():
    !git clone {REPO_URL}
    %cd Delphi

# install dependencies for the refactor (includes torch, numpy, pydantic, transformers)
!pip -q install -r delphi_torch/requirements.txt

# make refactor importable
import sys
sys.path.insert(0, 'delphi_torch/src')


## 2) Config
Adjust these settings for quicker or longer runs.


In [ ]:
import torch
from delphi_torch.config import DataConfig, ModelConfig, TrainConfig, AppConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'

data_cfg = DataConfig(
    dataset='ukb_simulated_data',
    batch_size=128 if device == 'cuda' else 32,
    block_size=48,
    num_workers=0,
    pin_memory=(device == 'cuda'),
)

model_cfg = ModelConfig(
    block_size=48,
    n_layer=12,
    n_head=12,
    d_model=120,
    dropout=0.1,
    token_dropout=0.0,
    vocab_size=1270,
    t_min=0.1,
)

train_cfg = TrainConfig(
    out_dir='Delphi-2M',
    max_steps=300,
    eval_interval=50,
    eval_iters=10,
    log_interval=10,
    learning_rate=2e-3,
    weight_decay=2e-1,
    beta2=0.99,
    warmup_steps=50,
    lr_decay_steps=300,
    min_lr=2e-4,
    device=device,
    dtype='float32',
)

cfg = AppConfig(data=data_cfg, model=model_cfg, train=train_cfg)
cfg


## 3) Data Pipeline (Dataset + DataLoader)


In [ ]:
from delphi_torch.data.dataloaders import build_dataloaders

train_loader, val_loader = build_dataloaders(cfg.data)
len(train_loader), len(val_loader)


## 4) Model + Loss


In [ ]:
from delphi_torch.models.delphi import DelphiModel
from delphi_torch.training.losses import compute_losses

model = DelphiModel(cfg.model).to(cfg.train.device)
model


## 5) Training + Evaluation
We track loss and token accuracy for train/eval.


In [ ]:
import math
from itertools import cycle

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.train.learning_rate,
    betas=(cfg.train.beta1, cfg.train.beta2),
    weight_decay=cfg.train.weight_decay,
)

def lr_at_step(step: int) -> float:
    if step < cfg.train.warmup_steps:
        return cfg.train.learning_rate * step / cfg.train.warmup_steps
    if step > cfg.train.lr_decay_steps:
        return cfg.train.min_lr
    decay_ratio = (step - cfg.train.warmup_steps) / (cfg.train.lr_decay_steps - cfg.train.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return cfg.train.min_lr + coeff * (cfg.train.learning_rate - cfg.train.min_lr)

def token_accuracy(logits, targets, ignore_tokens):
    preds = logits.argmax(-1)
    mask = targets != -1
    for t in ignore_tokens:
        mask &= targets != t
    if mask.sum() == 0:
        return float('nan')
    return (preds[mask] == targets[mask]).float().mean().item()

def eval_loop():
    model.eval()
    losses = []
    accs = []
    with torch.no_grad():
        for step, batch in enumerate(val_loader):
            x, a, y, b = [t.to(cfg.train.device) for t in batch]
            logits, attn_mask, _ = model(x, a, targets_age=b)
            loss_dict = compute_losses(
                logits, idx=x, age=a, targets=y, targets_age=b, attn_mask=attn_mask, cfg=cfg.model, validation=True
            )
            losses.append(loss_dict['loss'].item())
            accs.append(token_accuracy(logits, y, cfg.model.ignore_tokens))
            if step + 1 >= cfg.train.eval_iters:
                break
    model.train()
    return (sum(losses)/len(losses), sum(accs)/len(accs))

train_iter = cycle(train_loader)

steps, train_losses, val_losses, train_accs, val_accs = [], [], [], [], []

for step in range(1, cfg.train.max_steps + 1):
    x, a, y, b = next(train_iter)
    x, a, y, b = [t.to(cfg.train.device) for t in (x, a, y, b)]

    lr = lr_at_step(step)
    for group in optimizer.param_groups:
        group['lr'] = lr

    logits, attn_mask, _ = model(x, a, targets_age=b)
    loss_dict = compute_losses(
        logits, idx=x, age=a, targets=y, targets_age=b, attn_mask=attn_mask, cfg=cfg.model, validation=False
    )

    optimizer.zero_grad(set_to_none=True)
    loss_dict['loss'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip)
    optimizer.step()

    if step % cfg.train.log_interval == 0:
        acc = token_accuracy(logits, y, cfg.model.ignore_tokens)
        print(f"step {step}: loss {loss_dict['loss'].item():.4f} acc {acc:.4f}")

    if step % cfg.train.eval_interval == 0:
        val_loss, val_acc = eval_loop()
        steps.append(step)
        train_losses.append(loss_dict['loss'].item())
        val_losses.append(val_loss)
        train_accs.append(token_accuracy(logits, y, cfg.model.ignore_tokens))
        val_accs.append(val_acc)
        print(f"  eval: loss {val_loss:.4f} acc {val_acc:.4f}")


## 6) Training Curves


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps, train_losses, label='train')
axes[0].plot(steps, val_losses, label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('step')
axes[0].legend()

axes[1].plot(steps, train_accs, label='train')
axes[1].plot(steps, val_accs, label='val')
axes[1].set_title('Token Accuracy')
axes[1].set_xlabel('step')
axes[1].legend()

plt.show()


## 7) Inference Example (Next‑Event Prediction)
We show the last events for one sample and the model’s top‑5 next‑event predictions.


In [ ]:
import numpy as np

labels_path = Path('data/ukb_simulated_data/labels.csv')
labels = [line.strip() for line in labels_path.read_text().splitlines() if line.strip()]

# grab a batch from validation
x, a, y, b = next(iter(val_loader))
x, a = x.to(cfg.train.device), a.to(cfg.train.device)

with torch.no_grad():
    logits, _, _ = model(x, a)

sample_idx = 0
last_k = 8
tokens = x[sample_idx, -last_k:].cpu().numpy()
ages = a[sample_idx, -last_k:].cpu().numpy() / 365.25

print('Last events (token, label, age years):')
for t, age in zip(tokens, ages):
    label = labels[t] if t < len(labels) else f'#{t}'
    print(f'  {t:4d}  {label[:40]:40s}  {age:6.2f}')

# top‑k next prediction
logits_last = logits[sample_idx, -1].cpu()
topk = torch.topk(logits_last, k=5).indices.tolist()
print('\nTop‑5 predicted next tokens:')
for t in topk:
    label = labels[t] if t < len(labels) else f'#{t}'
    print(f'  {t:4d}  {label[:40]}')

# expected time to next event (rough proxy)
rate = torch.exp(torch.logsumexp(logits_last, dim=0))
expected_days = (1.0 / rate).item()
print(f'\nExpected time to next event (approx): {expected_days/365.25:.2f} years')


## 8) Full Trajectory Sampling
We sample a full continuation from a short seed trajectory using Delphi’s
competing‑exponentials mechanism.


In [ ]:
import torch

def sample_trajectory(
    model,
    seed_tokens,
    seed_ages,
    *,
    max_new_tokens=40,
    max_age_years=85,
    termination_tokens=None,
    no_repeat=True,
    ignore_tokens=None,
):
    model.eval()
    if termination_tokens is None:
        termination_tokens = []
    if ignore_tokens is None:
        ignore_tokens = []

    idx = seed_tokens.clone()
    age = seed_ages.clone()

    term = torch.tensor(termination_tokens, device=idx.device, dtype=torch.int64)

    for _ in range(max_new_tokens):
        logits, _, _ = model(idx, age)
        logits = logits[:, -1, :]
        if ignore_tokens:
            logits[:, ignore_tokens] = -torch.inf

        if no_repeat:
            fill = idx.clone()
            fill[fill == 1] = 0
            logits = logits.scatter_(1, fill, -torch.inf)

        # sample next event time from competing exponentials
        u = torch.rand_like(logits)
        t_next = -torch.exp(-logits) * torch.log(u)
        t_next = torch.clamp(t_next, min=0.0, max=365 * 80)
        dt, idx_next = t_next.min(1)
        age_next = age[:, -1] + dt

        idx = torch.cat([idx, idx_next[:, None]], dim=1)
        age = torch.cat([age, age_next[:, None]], dim=1)

        if term.numel() > 0:
            if torch.isin(idx, term).any(-1).all():
                break
        if (age_next > max_age_years * 365.25).all():
            break

    return idx, age

# seed from validation batch
x, a, y, b = next(iter(val_loader))
seed_len = 8
seed_tokens = x[:1, :seed_len].to(cfg.train.device)
seed_ages = a[:1, :seed_len].to(cfg.train.device)

sampled_tokens, sampled_ages = sample_trajectory(
    model,
    seed_tokens,
    seed_ages,
    max_new_tokens=30,
    max_age_years=85,
    termination_tokens=[],
    no_repeat=True,
    ignore_tokens=cfg.model.ignore_tokens,
)

tokens = sampled_tokens[0].cpu().numpy()
ages = sampled_ages[0].cpu().numpy() / 365.25

print(f'Sampled trajectory length: {len(tokens)}')
print('Last 12 events:')
for t, age in zip(tokens[-12:], ages[-12:]):
    label = labels[t] if t < len(labels) else f'#{t}'
    print(f'  {t:4d}  {label[:40]:40s}  {age:6.2f}')
